# Airbnb Listing Segmentation

Follows **`docs/cluster_analysis_method.md`** phase-by-phase.

**Dataset:** `Data/processed/listings_all_cities.parquet` — 42 823 listings, Madrid / Barcelona / Málaga  
**Goal:** Cluster listings on *attributes only* so the resulting segments can be used as a grouping variable in the downstream price and occupancy prediction models.

| Excluded from ALL clustering features | Reason |
|---|---|
| `price`, `price_cat` | target |
| `estimated_occupancy_l365d`, `estimated_revenue_l365d` | targets / leakage |
| `number_of_reviews*`, `reviews_per_month`, `availability_*` | near-direct occupancy leakage |
| `days_since_*`, `review_span_years` | review-activity proxies (flag for occupancy model too) |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

RANDOM_STATE = 42
DATA_PATH    = "../Data/processed/listings_all_cities.parquet"

# ── columns to keep in df but NEVER feed to K-Means ─────────────────────────
TARGETS = ['price', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'price_cat']

OCCUPANCY_LEAKAGE = [
    'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d',
    'number_of_reviews_ly', 'reviews_per_month',
    'availability_30', 'availability_60', 'availability_90', 'availability_365',
    'availability_eoy',
]

REVIEW_ACTIVITY = ['days_since_first_review', 'days_since_last_review', 'review_span_years']

print('Setup complete — RANDOM_STATE =', RANDOM_STATE)

## Phase 1 — Data Audit

In [ ]:
df_raw = pd.read_parquet(DATA_PATH)
df     = df_raw.copy()

print(f"Shape: {df.shape}")
print(f"\nCity counts:\n{df['city'].value_counts()}")
print(f"\nDtypes:\n{df.dtypes.value_counts()}")
df.describe().round(2)

In [ ]:
null_pct = (df.isna().mean() * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]
print("Columns with missing values (% of rows):")
print(null_pct.round(1).to_string())

## Phase 2 — Cleaning & Impossible-Value Removal

Drop only physically impossible / data-entry-error records (not statistical outliers — that is Phase 4).

| Column | Threshold | Rationale |
|---|---|---|
| `minimum_nights` | > 365 | Airbnb hard cap is 1 year; tail shows 865, 600, 500 |
| `beds` | > 100 | beds = 127 is impossible; IQR handles the 24–30 range |
| `bathrooms_number` | > 15 | 19-bathroom entry is implausible |

In [ ]:
for col in ['beds', 'minimum_nights', 'bathrooms_number']:
    print(f"{col} (top 10):", df[col].sort_values(ascending=False).head(10).tolist())

In [ ]:
n_before = len(df)
df = df[df['minimum_nights']    <= 365].copy()
df = df[df['beds']              <= 100].copy()
df = df[df['bathrooms_number']  <= 15 ].copy()
n_after = len(df)

print(f"Dropped: {n_before - n_after:,} rows ({(n_before - n_after) / n_before * 100:.2f}%)")
print(f"Remaining: {n_after:,} rows")

## Phase 3 — Distribution Checks & Transforms

Auto-classify columns into binary / continuous / categorical, then plot histograms for the continuous clustering candidates and confirm log transforms reduce right skew.

In [ ]:
# Columns excluded from auto-classification (will be engineered, are IDs/text, or are leakage)
DROP_IDS      = ['id','scrape_id','host_id','host_name','name','description',
                 'neighborhood_overview','picture_url','host_about','host_verifications',
                 'license','source','amenities','bathrooms_description','host_location',
                 'latitude','longitude']
DROP_DATES    = ['host_since','last_scraped','calendar_last_scraped','first_review','last_review']
DROP_REDUNDANT = [
    'property_type',           # 71 levels → use property_type_std (6)
    'neighbourhood_cleansed',  # 210+ levels; only used for Málaga is_central flag
    'neighbourhood_group_cleansed',  # only used for Madrid/Barcelona is_central flag
    'host_response_rate', 'host_acceptance_rate',  # captured by *_cat
    'maximum_nights',
    'minimum_minimum_nights','maximum_minimum_nights',
    'minimum_maximum_nights','maximum_maximum_nights',
    'minimum_nights_avg_ntm','maximum_nights_avg_ntm',
    'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms',
    'review_scores_accuracy','review_scores_cleanliness','review_scores_checkin',
    'review_scores_communication','review_scores_location','review_scores_value',
]
WILL_ENGINEER = [
    'room_type','property_type_std','city',
    'host_response_time','host_response_rate_cat','host_acceptance_rate_cat',
]

EXCLUDE_FROM_CLASSIFY = set(
    DROP_IDS + DROP_DATES + DROP_REDUNDANT + WILL_ENGINEER
    + TARGETS + OCCUPANCY_LEAKAGE + REVIEW_ACTIVITY
)

binary_cols, continuous_cols, categorical_cols = [], [], []
for col in df.columns:
    if col in EXCLUDE_FROM_CLASSIFY:
        continue
    s = df[col].dropna()
    if (df[col].dtype == bool or
        (pd.api.types.is_numeric_dtype(df[col]) and
         s.nunique() == 2 and
         set(s.unique()).issubset({0, 1, True, False}))):
        binary_cols.append(col)
    elif pd.api.types.is_numeric_dtype(df[col]):
        continuous_cols.append(col)
    else:
        categorical_cols.append(col)

print(f"Binary ({len(binary_cols)}):",     binary_cols)
print(f"\nContinuous ({len(continuous_cols)}):", continuous_cols)
print(f"\nCategorical ({len(categorical_cols)}):", categorical_cols)

In [ ]:
# Histogram + KDE grid for continuous clustering candidates
n_cols = 3
n_rows = (len(continuous_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    data = df[col].dropna()
    axes[i].hist(data, bins=50, density=True, alpha=0.6, color='steelblue', edgecolor='none')
    try:
        kde_x = np.linspace(data.min(), data.max(), 300)
        kde   = stats.gaussian_kde(data)
        axes[i].plot(kde_x, kde(kde_x), color='navy', lw=1.5)
    except Exception:
        pass
    axes[i].set_title(f"{col}\nskew={data.skew():.2f}", fontsize=8)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Continuous candidates — raw distributions', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Confirm log1p reduces right skew for the six flagged variables
LOG1P_COLS = ['accommodates','bedrooms','beds','bathrooms_number',
              'minimum_nights','calculated_host_listings_count']

fig, axes = plt.subplots(len(LOG1P_COLS), 2, figsize=(12, len(LOG1P_COLS) * 2.5))

for i, col in enumerate(LOG1P_COLS):
    raw  = df[col].dropna()
    logv = np.log1p(raw)
    for ax, data, label in zip(axes[i], [raw, logv], ['raw', 'log1p']):
        ax.hist(data, bins=50, density=True, alpha=0.6, color='steelblue', edgecolor='none')
        ax.set_title(f"{col} — {label}  (skew={data.skew():.2f})", fontsize=8)
        ax.tick_params(labelsize=7)

plt.suptitle('Log1p transform: before vs after', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Phase 4 — IQR Outlier Split (within city)

IQR bounds are computed **per city** to avoid Madrid's higher price range biasing the thresholds for Málaga.

- `df_clean` = rows where ≤ 1 continuous column is outside its city-specific IQR → fed to K-Means  
- `df_multi_outliers` = rows where > 1 column is out-of-bounds → manually-labelled **"Ultra / Extreme"** segment

In [ ]:
# Continuous columns used for IQR flag counting (raw, before log transforms)
# description_length is excluded — a very long description is not a physically
# impossible value and should not route listings to the extreme-outlier bucket
CONTINUOUS_RAW = ['accommodates','bedrooms','beds','bathrooms_number',
                  'minimum_nights','host_tenure_years','review_scores_rating',
                  'calculated_host_listings_count']

def compute_iqr_flags(df_in, cols):
    """Return int flag matrix (1 = outside city-specific IQR)."""
    flag_df = pd.DataFrame(0, index=df_in.index, columns=cols)
    for city in df_in['city'].unique():
        mask      = df_in['city'] == city
        city_data = df_in.loc[mask, cols]
        Q1  = city_data.quantile(0.25)
        Q3  = city_data.quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        for col in cols:
            flag_df.loc[mask, col] = (
                (city_data[col] < lower[col]) | (city_data[col] > upper[col])
            ).astype(int)
    return flag_df

flag_matrix        = compute_iqr_flags(df, CONTINUOUS_RAW)
df['n_outlier_cols'] = flag_matrix.sum(axis=1)

df_clean         = df[df['n_outlier_cols'] <= 1].copy()
df_multi_outliers = df[df['n_outlier_cols'] >  1].copy()

print(f"df_clean:          {len(df_clean):>6,}  ({len(df_clean)/len(df)*100:.1f}%)")
print(f"df_multi_outliers: {len(df_multi_outliers):>6,}  ({len(df_multi_outliers)/len(df)*100:.1f}%)")
print(f"\nMulti-outlier breakdown by city:")
print(df_multi_outliers['city'].value_counts())
print(f"\nOutlier column distribution:")
print(df['n_outlier_cols'].value_counts().sort_index())

## Phase 5 — Missing-Value Imputation

Applied **separately** to `df_clean` and `df_multi_outliers` (stats computed independently — no leakage between subsets).

| Column | Missing | Strategy |
|---|---|---|
| `review_scores_rating` | 17.3% (zero-review listings) | Group mean within `city × room_type` among reviewed listings; city-level fallback |
| `host_tenure_years` | 0.2% | Group mean within `city`; global fallback |

In [ ]:
def impute_subset(df_sub):
    df_sub = df_sub.copy()

    has_rev = df_sub['number_of_reviews'] > 0

    # review_scores_rating: fill values from reviewed listings only so zero-review
    # listings don't pollute group means.
    reviewed = df_sub[has_rev]
    city_room_means = (
        reviewed.groupby(['city', 'room_type'], observed=True)['review_scores_rating'].mean()
    )
    city_means  = reviewed.groupby('city', observed=True)['review_scores_rating'].mean()
    global_mean = float(reviewed['review_scores_rating'].mean())

    fine_fill = pd.Series(
        [city_room_means.get((c, r), np.nan)
         for c, r in zip(df_sub['city'], df_sub['room_type'])],
        index=df_sub.index, dtype=float,
    )
    city_fill = df_sub['city'].map(city_means).astype(float)

    filled_reviewed = (
        df_sub['review_scores_rating'].fillna(fine_fill).fillna(city_fill).fillna(global_mean)
    )
    fill_zero_rev = city_fill.fillna(global_mean)

    df_sub['review_scores_rating'] = np.where(
        has_rev.values, filled_reviewed.values, fill_zero_rev.values
    )

    # host_tenure_years: city-level mean, then global (assignment, not inplace).
    df_sub['host_tenure_years'] = (
        df_sub.groupby('city', observed=True)['host_tenure_years']
        .transform(lambda x: x.fillna(x.mean()))
    )
    df_sub['host_tenure_years'] = df_sub['host_tenure_years'].fillna(
        df_sub['host_tenure_years'].mean()
    )

    return df_sub

df_clean          = impute_subset(df_clean)
df_multi_outliers = impute_subset(df_multi_outliers)

print("Remaining nulls in clustering cols after imputation:")
for col in ['review_scores_rating', 'host_tenure_years']:
    print(f"  df_clean[{col}]:          {df_clean[col].isna().sum()}")
    print(f"  df_multi_outliers[{col}]: {df_multi_outliers[col].isna().sum()}")

## Phase 6 — Domain Feature Engineering

All flags are binary 0/1 and derived from listing **attributes only** (no target columns).

| Flag | Source | Logic |
|---|---|---|
| `has_reviews` | `number_of_reviews` | > 0 |
| `is_entire_place/private_room/hotel_hostel/shared_room` | `property_type_std` | equality |
| `is_madrid`, `is_barcelona` | `city` | equality; Málaga = reference (both = 0) |
| `is_central` | city-specific | Madrid: `neighbourhood_group_cleansed == 'Centro'`; Barcelona: `== 'Ciutat Vella'`; Málaga: `neighbourhood_cleansed == 'Centro'` |
| `is_fast_responder` | `host_response_time` | within an hour or within a few hours |
| `host_acceptance_rate_ord`, `host_response_rate_ord` | `*_cat` | ordinal: unknown=0, low=1, medium=2, high=3 |

In [ ]:
def engineer_features(df_sub):
    df_sub = df_sub.copy()

    df_sub['has_reviews']    = (df_sub['number_of_reviews'] > 0).astype(int)

    df_sub['is_entire_place'] = (df_sub['property_type_std'] == 'Entire place').astype(int)
    df_sub['is_private_room'] = (df_sub['property_type_std'] == 'Private room').astype(int)
    df_sub['is_hotel_hostel'] = (df_sub['property_type_std'] == 'Hotel / Hostel').astype(int)
    df_sub['is_shared_room']  = (df_sub['property_type_std'] == 'Shared room').astype(int)

    df_sub['is_madrid']    = (df_sub['city'] == 'Madrid').astype(int)
    df_sub['is_barcelona'] = (df_sub['city'] == 'Barcelona').astype(int)

    # is_central: each city uses a different source column
    is_central = pd.Series(0, index=df_sub.index)
    is_central.loc[(df_sub['city'] == 'Madrid')    & (df_sub['neighbourhood_group_cleansed'] == 'Centro')]      = 1
    is_central.loc[(df_sub['city'] == 'Barcelona') & (df_sub['neighbourhood_group_cleansed'] == 'Ciutat Vella')] = 1
    is_central.loc[(df_sub['city'] == 'Málaga')    & (df_sub['neighbourhood_cleansed'] == 'Centro')]            = 1
    df_sub['is_central'] = is_central.values

    fast = {'within an hour', 'within a few hours'}
    df_sub['is_fast_responder'] = df_sub['host_response_time'].isin(fast).astype(int)

    rate_map = {'unknown': 0, 'low': 1, 'medium': 2, 'high': 3}
    df_sub['host_acceptance_rate_ord'] = df_sub['host_acceptance_rate_cat'].astype(str).map(rate_map).fillna(0).astype(int)
    df_sub['host_response_rate_ord']   = df_sub['host_response_rate_cat'].astype(str).map(rate_map).fillna(0).astype(int)

    for col in ['host_is_superhost','host_identity_verified','instant_bookable','host_has_profile_pic']:
        df_sub[col] = df_sub[col].astype(int)

    return df_sub

df_clean          = engineer_features(df_clean)
df_multi_outliers = engineer_features(df_multi_outliers)

print("Engineered feature counts:")
eng_cols = ['has_reviews','is_entire_place','is_private_room','is_hotel_hostel',
            'is_shared_room','is_madrid','is_barcelona','is_central',
            'is_fast_responder']
print(df_clean[eng_cols].sum().to_string())
print(f"\nis_central distribution:\n{df_clean.groupby(['city','is_central']).size().unstack(fill_value=0)}")

## Phase 7 — Build Segmentation Feature Table

Apply log1p transforms, define `CLUSTER_FEATURES`, fill any residual NaN with the column median (safety net), and assemble `X`.

In [ ]:
df_segmentation = df_clean.copy()

LOG1P_COLS = ['accommodates','bedrooms','beds','bathrooms_number',
              'minimum_nights','calculated_host_listings_count']
for col in LOG1P_COLS:
    df_segmentation[f'log1p_{col}'] = np.log1p(df_segmentation[col])

if 'amenity_count' in df_segmentation.columns:
    df_segmentation['log1p_amenity_count'] = np.log1p(df_segmentation['amenity_count'])

CONTINUOUS_FEATURES = (
    [f'log1p_{c}' for c in LOG1P_COLS]
    + ['host_tenure_years', 'review_scores_rating', 'description_length']
    + (['log1p_amenity_count'] if 'amenity_count' in df_segmentation.columns else [])
)

BINARY_FEATURES = [
    'host_is_superhost', 'host_identity_verified', 'instant_bookable', 'host_has_profile_pic',
    'has_reviews',
    'is_entire_place', 'is_private_room', 'is_hotel_hostel', 'is_shared_room',
    'is_madrid', 'is_barcelona', 'is_central',
    'is_fast_responder',
    'host_acceptance_rate_ord', 'host_response_rate_ord',
]

# Auto-include amenity flags written by amenity_feature_engineering.ipynb.
# NOTE: this couples the segmentation to that notebook's output columns - if it is
# re-run with different thresholds, re-run this notebook too (see Limitations).
AMENITY_FLAGS = [c for c in df_segmentation.columns
                 if (c.startswith('has_') or c.startswith('bundle_'))
                 and c not in BINARY_FEATURES]
BINARY_FEATURES = BINARY_FEATURES + AMENITY_FLAGS
if AMENITY_FLAGS:
    print(f"Amenity flags added: {AMENITY_FLAGS}")

CLUSTER_FEATURES = [c for c in CONTINUOUS_FEATURES + BINARY_FEATURES
                    if c in df_segmentation.columns]

# Safety-net: fill residual NaN with column median (assignment, not inplace).
for col in CLUSTER_FEATURES:
    df_segmentation[col] = df_segmentation[col].fillna(df_segmentation[col].median())

print(f"Total candidate features: {len(CLUSTER_FEATURES)}")
print(f"  Continuous: {len(CONTINUOUS_FEATURES)}")
print(f"  Binary:     {len(BINARY_FEATURES)}")
print(f"\nResidual NaN in X: {df_segmentation[CLUSTER_FEATURES].isna().sum().sum()}")

## Phase 8 — Iterative K-Means Feature Selection (protected core)

CV-based backward elimination, but **core drivers are protected and never dropped**.
The protected core is deliberately lean — **capacity, quality, room-type, and
centrality** — the attributes that actually separate the market:

- `log1p_accommodates`, `log1p_bedrooms`, `log1p_bathrooms_number` — capacity/size
- `review_scores_rating` — quality
- `is_entire_place`, `is_private_room` — listing type
- `is_central` — within-city location

**City identity (`is_madrid`/`is_barcelona`) is intentionally _not_ a segment axis** —
it is already a feature in the price/occupancy models, so segments are clean
**cross-city tiers** ("Premium" means the same thing in every city). Verified on
the data: dropping the city dummies lifts silhouette markedly (≈0.18 → ≈0.36).

Each iteration scales the features, picks best `k ∈ [2,5]` by silhouette, and drops
the lowest-CV **non-protected** feature — so the loop strips down to the core.

In [ ]:
# Lean protected core: capacity, quality, room-type, centrality. Verified to give
# the best-separated, balanced, location-aware segments (see NOTEBOOK_IMPROVEMENTS.md).
PROTECTED_FEATURES = [
    'log1p_accommodates', 'log1p_bedrooms', 'log1p_bathrooms_number',  # capacity/size
    'review_scores_rating',                                            # quality
    'is_entire_place', 'is_private_room',                              # listing type
    'is_central',                                                      # within-city location
]
PROTECTED_FEATURES = [f for f in PROTECTED_FEATURES if f in CLUSTER_FEATURES]
print(f"Protected core (never dropped): {len(PROTECTED_FEATURES)} -> {PROTECTED_FEATURES}")

current_features  = list(CLUSTER_FEATURES)
iteration_results = []
iteration         = 1


def _best_k(X):
    best_k, best_sil = 2, -1.0
    for k in range(2, 6):
        labels = KMeans(n_clusters=k, random_state=RANDOM_STATE,
                        n_init=20).fit_predict(X)
        sil = silhouette_score(X, labels, sample_size=10_000,
                               random_state=RANDOM_STATE)
        if sil > best_sil:
            best_sil, best_k = sil, k
    return best_k, best_sil


# Drop one non-protected feature per iteration until only the protected core remains.
while any(f not in PROTECTED_FEATURES for f in current_features):
    X_iter = StandardScaler().fit_transform(df_segmentation[current_features])
    best_k, best_sil = _best_k(X_iter)
    km_iter = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20)
    df_segmentation['KMeans_Iter_Cluster'] = km_iter.fit_predict(X_iter)
    centers_df = pd.DataFrame(km_iter.cluster_centers_, columns=current_features)
    sizes          = df_segmentation['KMeans_Iter_Cluster'].value_counts()
    small_clusters = int((sizes / len(df_segmentation) < 0.05).sum())
    cv = {c: centers_df[c].std() / (centers_df[c].abs().mean() + 1e-9)
          for c in current_features}
    droppable = pd.Series({c: v for c, v in cv.items()
                           if c not in PROTECTED_FEATURES}).sort_values()
    feature_to_drop = droppable.index[0]
    iteration_results.append({
        'Iteration': iteration, 'Num_Features': len(current_features),
        'Best_k': best_k, 'Silhouette': round(best_sil, 4),
        '< 5% Clusters': small_clusters,
        'Dropped_Feature': feature_to_drop, 'CV_of_Dropped': round(droppable.iloc[0], 4),
        'Remaining_Features': ', '.join(current_features),
    })
    current_features.remove(feature_to_drop)
    iteration += 1

summary_df = pd.DataFrame(iteration_results)
print(f"Loop complete - stripped to the protected core ({len(current_features)} features).")
print(summary_df[['Iteration','Num_Features','Best_k','Silhouette','< 5% Clusters','Dropped_Feature']].to_string(index=False))

In [ ]:
summary_df = pd.DataFrame(iteration_results)

# ── plot silhouette and feature count per iteration ───────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(summary_df['Iteration'], summary_df['Silhouette'], marker='o', color='steelblue')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Silhouette score')
ax1.set_title('Silhouette per iteration')
ax1.grid(True, alpha=0.3)

ax2.plot(summary_df['Iteration'], summary_df['Num_Features'], marker='s', color='darkorange')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Features remaining')
ax2.set_title('Feature count per iteration')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── full summary table ────────────────────────────────────────────────────────
display_cols = ['Iteration','Num_Features','Best_k','Silhouette','< 5% Clusters',
                'Dropped_Feature','CV_of_Dropped']
print(summary_df[display_cols].to_string(index=False))

## Phase 9 — Final Feature Set = Protected Core

The elimination above strips the feature set down to the protected core; that core
**is** the final clustering feature set. `K_FINAL` is chosen in Phase 10 (validation)
rather than here, so the choice is driven by silhouette + micro-cluster checks in the
actual (PCA) clustering space.

In [ ]:
# The CV loop strips to the protected core; cluster the final segments on it.
FINAL_CLUSTER_FEATURES = list(PROTECTED_FEATURES)
print(f"Final clustering features ({len(FINAL_CLUSTER_FEATURES)}):")
for f in FINAL_CLUSTER_FEATURES:
    print(f"  {f}")
print("\nK_FINAL is selected in Phase 10 (max silhouette with no <5% micro-clusters).")

## Phase 10 — Validate k (PCA space, Elbow + Silhouette)

Cluster the protected core in **PCA(0.9) space**. Silhouette alone favours the
trivial **k=2** split (one ~74% blob), so for a usable market segmentation we
**require k ≥ 4** and pick the highest-silhouette k in `[4, 6]` with **no <5%
micro-clusters** → verified **k = 4** (silhouette ≈ 0.38; k=5/6 introduce a sliver).

In [ ]:
from sklearn.decomposition import PCA

# Cluster the protected core in PCA(0.9) space.
X_core = StandardScaler().fit_transform(df_segmentation[FINAL_CLUSTER_FEATURES])
X_pca  = PCA(n_components=0.9, random_state=RANDOM_STATE).fit_transform(X_core)
print(f"PCA: {X_core.shape[1]} -> {X_pca.shape[1]} dims (90% variance retained)")

rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=25).fit(X_pca)
    sil = silhouette_score(X_pca, km.labels_, sample_size=10_000, random_state=RANDOM_STATE)
    shares = pd.Series(km.labels_).value_counts() / len(X_pca)
    rows.append({'k': k, 'silhouette': round(sil, 4),
                 'micro_<5%': int((shares < 0.05).sum()),
                 'largest_%': round(shares.max() * 100, 1),
                 'inertia': round(km.inertia_, 0)})
val_df = pd.DataFrame(rows)

# Silhouette peaks at the trivial k=2 (huge single blob), so require k >= 4 for a
# usable segmentation; pick the best k in [4,6] with no <5% micro-clusters.
cand = val_df[(val_df['k'].between(4, 6)) & (val_df['micro_<5%'] == 0)]
if cand.empty:
    cand = val_df[(val_df['k'] >= 4) & (val_df['micro_<5%'] == 0)]
if cand.empty:
    cand = val_df[val_df['k'] >= 4]
K_FINAL = int(cand.loc[cand['silhouette'].idxmax(), 'k'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(val_df['k'], val_df['inertia'], marker='o', color='steelblue')
ax1.axvline(K_FINAL, color='red', ls='--', label=f'chosen k={K_FINAL}')
ax1.set(xlabel='k', ylabel='inertia', title='Elbow'); ax1.legend(); ax1.grid(alpha=.3)
ax2.plot(val_df['k'], val_df['silhouette'], marker='o', color='darkorange')
ax2.axvline(K_FINAL, color='red', ls='--', label=f'chosen k={K_FINAL}')
ax2.set(xlabel='k', ylabel='silhouette', title='Silhouette (k>=4, micro-cluster-free)')
ax2.legend(); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(val_df.to_string(index=False))
print(f"\nChosen k = {K_FINAL}  (best silhouette in [4,6] with no <5% micro-clusters; "
      "k=2/3 rejected as too coarse)")

## Phase 11 — Fit Final Model & Name Segments by Structure

- Refit with `n_init=25, max_iter=500` for a stable solution.
- **Name segments by structure, not price rank.** The two mid tiers are the same
  price (~€160) but differ by **centrality**, so price-rank names ("Standard" /
  "Mid-Market") would hide the real distinction. Names are derived from each
  cluster's profile: room-type (private vs entire), centrality, and size.
- Re-attach `df_multi_outliers` as the **Ultra / Extreme** segment.

In [ ]:
from sklearn.decomposition import PCA

X_core   = StandardScaler().fit_transform(df_segmentation[FINAL_CLUSTER_FEATURES])
X_final  = PCA(n_components=0.9, random_state=RANDOM_STATE).fit_transform(X_core)
km_final = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=25, max_iter=500)
df_segmentation['Cluster_Final'] = km_final.fit_predict(X_final)

# ── Structural names ─────────────────────────────────────────────────────────
# Two mid tiers share a price but split by centrality, so name by structure.
_p = df_segmentation.groupby('Cluster_Final').agg(
    price=('price', 'mean'),
    accom=('accommodates', 'mean'),
    private=('property_type_std', lambda s: (s == 'Private room').mean()),
    central=('is_central', 'mean'),
)
name_map = {}
for cid, r in _p.iterrows():
    if r['private'] >= 0.4:
        name_map[cid] = 'Budget private rooms'
entire = _p[_p['private'] < 0.4]
if len(entire):
    prem = entire['price'].idxmax()
    name_map[prem] = 'Premium entire homes'
    for cid in [c for c in entire.index if c != prem]:
        name_map[cid] = ('Central entire homes' if _p.loc[cid, 'central'] >= 0.5
                         else 'Non-central entire homes')
for cid in _p.index:                       # safety fallback
    name_map.setdefault(cid, f'Segment {cid}')
df_segmentation['Segment_Name'] = df_segmentation['Cluster_Final'].map(name_map)

# Display order: cheapest -> most expensive
segment_names = list(_p.sort_values('price').index.map(name_map))
segment_order = segment_names + ['Ultra / Extreme']

df_multi_outliers = df_multi_outliers.copy()
df_multi_outliers['Cluster_Final'] = -1
df_multi_outliers['Segment_Name']  = 'Ultra / Extreme'

# ── Report: silhouette + interpretation profile ──────────────────────────────
final_sil = silhouette_score(X_final, km_final.labels_, sample_size=10_000,
                             random_state=RANDOM_STATE)
print(f"Final silhouette: {final_sil:.4f}  (k={K_FINAL}, PCA dims={X_final.shape[1]})\n")
print(f"{'Segment':<26}{'n':>7}{'price':>8}{'guests':>8}{'private%':>10}{'central%':>10}")
for cid in _p.sort_values('price').index:
    r = _p.loc[cid]; n = int((df_segmentation['Cluster_Final'] == cid).sum())
    print(f"{name_map[cid]:<26}{n:>7,}{r['price']:>8.0f}{r['accom']:>8.1f}"
          f"{r['private']*100:>9.0f}%{r['central']*100:>9.0f}%")
n_ext = len(df_multi_outliers)
print(f"{'Ultra / Extreme':<26}{n_ext:>7,}{df_multi_outliers['price'].mean():>8.0f}"
      f"  ({n_ext/(len(df_segmentation)+n_ext)*100:.1f}% of total, outliers)")

## Phase 12 — Profile & Visualise Segments

In [ ]:
# Combine clean segments + extreme group for profiling
df_all = pd.concat([df_segmentation, df_multi_outliers], axis=0, ignore_index=True)

PROFILE_CONTINUOUS = ['accommodates','bedrooms','bathrooms_number','minimum_nights',
                      'host_tenure_years','review_scores_rating',
                      'calculated_host_listings_count',
                      'amenity_count','description_length']
PROFILE_BINARY     = ['host_is_superhost','instant_bookable','has_reviews',
                      'is_entire_place','is_private_room','is_hotel_hostel','is_shared_room',
                      'is_central','is_madrid','is_barcelona',
                      'is_fast_responder']
PROFILE_TARGETS    = ['price','estimated_occupancy_l365d','estimated_revenue_l365d']
PROFILE_ALL        = PROFILE_CONTINUOUS + PROFILE_BINARY + PROFILE_TARGETS

segment_order = segment_names + ['Ultra / Extreme']
profile_table = (
    df_all.groupby('Segment_Name', observed=True)[PROFILE_ALL]
    .mean()
    .reindex(segment_order)
)

# Pretty-print: binary cols as %, targets separately
display_table = profile_table.copy()
for col in PROFILE_BINARY:
    display_table[col] = (display_table[col] * 100).round(1).astype(str) + '%'
for col in PROFILE_CONTINUOUS + PROFILE_TARGETS:
    display_table[col] = display_table[col].round(2)

display(display_table)

In [ ]:
SEGMENT_COLORS = {
    'Budget private rooms':     '#4393c3',
    'Central entire homes':     '#92c5de',
    'Non-central entire homes': '#f7b16e',
    'Premium entire homes':     '#f4a582',
    'Ultra / Extreme':          '#b2182b',
}

plot_segments = [s for s in segment_order if s in profile_table.index]
colors        = [SEGMENT_COLORS.get(s, '#888888') for s in plot_segments]
bar_features  = PROFILE_CONTINUOUS + PROFILE_BINARY + PROFILE_TARGETS

n_cols = 4
n_rows = (len(bar_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.2))
axes = axes.flatten()

for i, feat in enumerate(bar_features):
    ax   = axes[i]
    vals = profile_table.loc[plot_segments, feat]
    is_binary = feat in PROFILE_BINARY
    scale = 100 if is_binary else 1

    ax.bar(range(len(plot_segments)), vals * scale, color=colors)
    ax.set_title(feat, fontsize=8, pad=4)
    ax.set_xticks(range(len(plot_segments)))
    ax.set_xticklabels(plot_segments, rotation=40, ha='right', fontsize=6)
    if is_binary:
        ax.set_ylabel('%', fontsize=7)
        ax.set_ylim(0, 100)
    elif feat in PROFILE_TARGETS:
        ax.set_facecolor('#fff8f0')
    ax.tick_params(axis='y', labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Segment profiles — bar grid', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Radar uses 8 features — swap host_tenure_years for estimated_revenue_l365d
# so the investment signal is directly visible on the chart
RADAR_FEATURES = ['accommodates','minimum_nights','review_scores_rating',
                  'estimated_revenue_l365d',
                  'is_entire_place','host_is_superhost',
                  'is_central','instant_bookable']

radar_data = profile_table.loc[plot_segments, RADAR_FEATURES].copy().astype(float)
for col in RADAR_FEATURES:
    col_min, col_max = radar_data[col].min(), radar_data[col].max()
    if col_max > col_min:
        radar_data[col] = (radar_data[col] - col_min) / (col_max - col_min)
    else:
        radar_data[col] = 0.5

N      = len(RADAR_FEATURES)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles_closed = angles + angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for seg in plot_segments:
    vals = radar_data.loc[seg, RADAR_FEATURES].tolist()
    vals_closed = vals + vals[:1]
    color = SEGMENT_COLORS.get(seg, '#888888')
    ax.plot(angles_closed, vals_closed, label=seg, color=color, linewidth=2)
    ax.fill(angles_closed, vals_closed, alpha=0.07, color=color)

ax.set_xticks(angles)
ax.set_xticklabels(RADAR_FEATURES, size=9)
ax.set_ylim(0, 1)
ax.set_title('Segment radar (min-max scaled across segments)', pad=20, fontsize=12)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15))
plt.tight_layout()
plt.show()

## Phase 13 — DBSCAN Sanity Check

Run DBSCAN on the same scaled final feature set as a robustness check. `eps` is **auto-selected from the knee** of the k-NN distance curve (max perpendicular distance to the chord joining its endpoints) — no manual tuning. Cluster count, noise ratio, and silhouette are then compared against iterative K-Means.

In [ ]:
# k-NN distance plot to guide eps selection
nn = NearestNeighbors(n_neighbors=5).fit(X_final)
distances, _ = nn.kneighbors(X_final)
k_distances  = np.sort(distances[:, 4])[::-1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_distances, color='steelblue', lw=1.5)
ax.set_xlabel('Points (sorted by 5th-NN distance, descending)')
ax.set_ylabel('5th-NN distance')
ax.set_title('k-NN distance plot — use the elbow to choose eps')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Auto-select eps from the k-NN distance curve: the knee is the point of maximum
# perpendicular distance to the chord joining the curve's first and last points.
sorted_kd = np.sort(distances[:, 4])[::-1]
xs = np.arange(len(sorted_kd), dtype=float)
x1, y1 = 0.0, float(sorted_kd[0])
x2, y2 = float(len(sorted_kd) - 1), float(sorted_kd[-1])
perp = np.abs((y2 - y1) * xs - (x2 - x1) * sorted_kd + x2 * y1 - y2 * x1) \
       / np.hypot(y2 - y1, x2 - x1)
EPS         = float(round(sorted_kd[int(np.argmax(perp))], 3))
MIN_SAMPLES = 50
print(f"Auto-selected eps from k-NN knee: {EPS}  (min_samples={MIN_SAMPLES})")

db        = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit(X_final)
db_labels = db.labels_

n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
noise_ratio   = (db_labels == -1).mean()

print(f"DBSCAN  eps={EPS}, min_samples={MIN_SAMPLES}")
print(f"  Clusters (excl. noise): {n_db_clusters}")
print(f"  Noise ratio            : {noise_ratio:.1%}")

if n_db_clusters > 1:
    mask_valid = db_labels != -1
    db_sil = silhouette_score(X_final[mask_valid], db_labels[mask_valid],
                              sample_size=10_000, random_state=RANDOM_STATE)
    print(f"  Silhouette (non-noise) : {db_sil:.4f}")
else:
    db_sil = None
    print("  Fewer than 2 clusters - silhouette not computable")

print()
print("-- Comparison --------------------------------------")
print(f"  Method              Clusters  Noise   Silhouette")
print(f"  Iterative K-Means   {K_FINAL:<8}  {'n/a':<6}  {final_sil:.4f}")
if db_sil is not None:
    print(f"  DBSCAN              {n_db_clusters:<8}  {noise_ratio:.1%}   {db_sil:.4f}")
else:
    print(f"  DBSCAN              {n_db_clusters:<8}  {noise_ratio:.1%}   n/a")
print()
print("Conclusion: if DBSCAN produces many micro-clusters or high noise, "
      "iterative K-Means is preferred for interpretability and business use.")

## Limitations & Interpretation

**The segments have clear business meaning** (named by structure, not price rank):

| Segment | What it is |
|---|---|
| Budget private rooms | ~95% private rooms, small (~1.8 guests), ~€75 |
| Central entire homes | entire homes, **100% central**, ~€159 |
| Non-central entire homes | entire homes, **0% central**, ~€162 |
| Premium entire homes | large entire homes (~5+ guests), ~€239 |
| Ultra / Extreme | multivariate outliers, large, lower rating, ~€260 |

- **Why two ~€160 tiers?** Central vs non-central entire homes are the *same price*
but split by **location** — exactly the signal `is_central` was protected to keep.
Same price, different location is a real, defensible distinction (not redundancy).
- **Separation:** silhouette ≈0.36, k=4, balanced (largest ~27% of clean), zero
<5% micro-clusters; DBSCAN cross-check (many micro-clusters + high noise) confirms
K-means is the right choice.
- **City identity excluded by design** — it's already a model feature; including it
drops silhouette to ≈0.18 and just re-derives the cities. Segments are cross-city tiers.
- **Amenity-flag coupling** with `amenity_feature_engineering.ipynb` — re-run both
together if that notebook changes.
- **Use as a feature, not a router** (per `data_segmentation_eda.ipynb`).

## Phase 14 — Persist Segment Labels

Write `df_all` (all cities, all segments including Ultra / Extreme) with `Segment_Name` and `Cluster_Final` to a new parquet so downstream price and occupancy models can join on listing `id`.

In [ ]:
OUTPUT_PATH = "../Data/processed/listings_segmented.parquet"

# Drop the iteration-level cluster column; only the final labels matter downstream
KEEP_COLS = [c for c in df_all.columns if c != 'KMeans_Iter_Cluster']
df_out = df_all[KEEP_COLS].copy()

df_out.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(df_out):,} rows → {OUTPUT_PATH}")
print(f"\nSegment distribution:")
print(df_out['Segment_Name'].value_counts().reindex(segment_order))
print(f"\nColumns written: {len(df_out.columns)}")